In [1]:
import functions as fn
from mastu import HSV
import numpy as np
import sys

import matplotlib.pyplot as plt
%matplotlib widget

In [2]:
dictFile = '/home/sthoma/Documents/Results/rba/50499/rba_results_invert_6.txt'
inputDict = fn.loadDict(dictFile)
loadFile = dictFile.replace('.txt', '.npz')

### unzip the dictionary
shotn = inputDict['shotn']
Rzfile = inputDict['Rzfile']

### the inputs to select which bits of video, and averaging
I0 = inputDict['I0']
I1 = inputDict['I1']
J0 = inputDict['J0']
J1 = inputDict['J1']
tend = inputDict['tend']
raverage = inputDict['raverage']
taverage = inputDict['taverage']
T0 = inputDict['T0']
T1 = inputDict['T1']

### for converting to photons and account for vignetting
photons = bool(inputDict['photons'])
exposureMult = inputDict['exposureMult']
vignette = bool(inputDict['vignette'])

### variables for the inversion, should be approximately constant
saveFile = inputDict['saveFile']
rEnd = inputDict['rEnd']
nrMult = inputDict['nrMult']
sysErr = inputDict['sysErr']
biasedEdges = bool(inputDict['biasedEdges'])
nFisher = inputDict['nFisher']
regGuess = inputDict['regGuess']
regMin = inputDict['regMin']

### use pedestal fitting or raw Thomson
kindThomson = inputDict['kindThomson']
### R value pedestal fitting works up to
Rprofile0 = inputDict['Rprofile0']

### kind of interpolation to use, don't need to change
line = inputDict['line']
kindExcite = inputDict['kindExcite']
kindRecomb = inputDict['kindRecomb']
kindIonise = inputDict['kindIonise']

### using a constant percentage for profile error
percent = inputDict['percent']

### temperature to assume for neutral density from fig
figTemp = inputDict['figTemp']

### load the results from inversion
inversion = np.load(loadFile)
R = inversion['R']
data = inversion['data']
nT, nR = data.shape
err = inversion['err']
mask = inversion['mask']
time = inversion['time']
Rgrid = inversion['Rgrid']
RgridB = inversion['RgridB']
emissivity = inversion['emissivity']
emissivityErr = inversion['emissivityErr']
backprojection = inversion['backprojection']
scale = inversion['scale']
emMax = inversion['emMax']
emMaxR = inversion['emMaxR']
emR0 = inversion['emR0']
emR1 = inversion['emR1']
emFWHM = inversion['emFWHM']
emFWHMerr = inversion['emFWHMerr']
Rind = inversion['Rind']
Rprofile = inversion['Rprofile']

In [7]:
kindThomson

'raw'

In [3]:
### using raw Thomson or Pedestal fitting
if kindThomson == 'fit':
    ### load the pedestal fitting parameters
    timeProfile, R0Density, heightDensity, widthDensity, \
        gradDensity, bkgdDensity = HSV.getPedestal(shotn, 'n_e')
    _, R0Temp, heightTemp, widthTemp, \
        gradTemp, bkgdTemp = HSV.getPedestal(shotn, 'T_e')

### always load the Thomson data
timeThomson, dataDensity, _, Rthomson = HSV.getThomson(shotn, 'n_e')
_, dataTemp, _, _ = HSV.getThomson(shotn, 'T_e')

### make ADAS data functions
### TODO: Extrapolation arguments are hardcoded
fExcite, scaleExcite, fRecomb, scaleRecomb, fIonise, scaleIonise = fn.makeADAS(
    line=line, excite=kindExcite, recomb=kindRecomb, 
    ionise=kindIonise, bounds_error=False, fill_value=None
)

In [5]:
dR = np.linspace(-0.024, 0.024, 25)
NdR = len(dR)

### make empty arrays for Thomson profiles
profileTemp = np.zeros((nT, len(Rprofile), NdR))
profileDensity = np.zeros((nT, len(Rprofile), NdR))
### where the end of the Thomson data is
RendThomson = np.zeros(nT)
### make empty arrays for Siz, n0, and errors
ioniseRate = np.zeros((nT, len(Rprofile), NdR))
ioniseErr = np.zeros((nT, len(Rprofile), NdR))
neutralDensity = np.zeros((nT, len(Rprofile), NdR))
neutralErr = np.zeros((nT, len(Rprofile), NdR))
### empty arrays for measurements of the ionisation profile
ioniseMax = np.zeros((nT, NdR))
ioniseMaxR = np.zeros((nT, NdR))
ioniseR0 = np.zeros((nT, NdR))
ioniseR1 = np.zeros((nT, NdR))
ioniseFWHM = np.zeros((nT, NdR))
ioniseFWHMerr = np.zeros((nT, NdR))
### empty arrays for measurements of the neutral profile
neutralMax = np.zeros((nT, NdR))
neutralMaxR = np.zeros((nT, NdR))
### here, I use exponential lengths instead of FWHM
neutralR1 = np.zeros((nT, NdR))
neutralR2 = np.zeros((nT, NdR))
neutralR3 = np.zeros((nT, NdR))
neutralWidth1 = np.zeros((nT, NdR))
neutralWidth1err = np.zeros((nT, NdR))
neutralWidth2 = np.zeros((nT, NdR))
neutralWidth2err = np.zeros((nT, NdR))
neutralWidth3 = np.zeros((nT, NdR))
neutralWidth3err = np.zeros((nT, NdR))
neutralR1A = np.zeros((nT, NdR))
neutralR2A = np.zeros((nT, NdR))
neutralR3A = np.zeros((nT, NdR))
neutralWidth1A = np.zeros((nT, NdR))
neutralWidth1Aerr = np.zeros((nT, NdR))
neutralWidth2A = np.zeros((nT, NdR))
neutralWidth2Aerr = np.zeros((nT, NdR))
neutralWidth3A = np.zeros((nT, NdR))
neutralWidth3Aerr = np.zeros((nT, NdR))
### empty arrays for psiN, fig pressure, and separatrix ratio
psiN = np.zeros((nT, len(Rprofile), NdR))
figN0 = np.zeros((nT, NdR))
neutralRatio = np.zeros((nT, NdR))

### iterate over the times
for i in range(nT):
    for j in range(0, NdR):

        ### make profiles for this timestep
        T = fn.findNearest(timeThomson, time[i])

        ### find where the last Thomson data point is
        ### don't use HFS, LFS only
        rTh = fn.findNearest(Rthomson[T], 1.) # TODO, hardcoded 1.
        xTh = Rthomson[T,rTh:]
        ### get rid of Infs and NaNs
        booThomson = np.isfinite(dataTemp[T,rTh:]) * \
                    np.isfinite(dataDensity[T,rTh:])
        ### where the last datapoint is
        RendThomson[i] = xTh[booThomson][-1]


        ### handle the pedestalFit or raw data differently
        if kindThomson == 'fit':

            TT = fn.findNearest(timeProfile, time[i])
            ### using pedestal fitting results
            profileTemp[i,:,j] = HSV.mtanh(
                Rprofile+dR[j], R0Temp[TT], heightTemp[TT],
                widthTemp[TT], gradTemp[TT], bkgdTemp[TT]
            )
            profileDensity[i,:,j] = HSV.mtanh(
                Rprofile+dR[j], R0Density[TT], heightDensity[TT],
                widthDensity[TT], gradDensity[TT], bkgdDensity[TT]
            )

        elif kindThomson == 'raw':
            ### using the raw Thomson data

            ### temperature
            yTh = dataTemp[T,rTh:][booThomson]
            left = None
            right = None
            # right = 0.200001
            profileTemp[i,:,j] = np.interp(
                Rprofile, xTh[booThomson]+dR[j], yTh, left=left, right=right
                )

            ### density
            yTh = dataDensity[T,rTh:][booThomson]
            left = None
            right = None
            # right = 5.000001e13
            profileDensity[i,:,j] = np.interp(
                Rprofile, xTh[booThomson]+dR[j], yTh, left=left, right=right
                )
            ### TODO: hardcoded extrapolation method

        ### throw it into the iteration function
        ioniseRate[i,:,j], ioniseErr[i,:,j], neutralDensity[i,:,j], neutralErr[i,:,j] = \
            fn.neutrals(
                emissivity[i,Rind:], emissivityErr[i,Rind:], profileTemp[i,:,j],
                profileDensity[i,:,j], 
                fExcite, scaleExcite, fRecomb, scaleRecomb,
                fIonise, scaleIonise, ni=1.0
        )

        ### find the emissivity maximum and location
        ioniseMax[i,j] = ioniseRate[i,:,j].max()
        ioniseMaxR[i,j] = Rprofile[ioniseRate[i,:,j].argmax()]

        ### calculate the ionisation FWHM
        ioniseR0[i,j], ioniseR1[i,j] = fn.findPosition(
            Rprofile, ioniseRate[i,:,j], 0.5, kind='linear'
        )
        ioniseFWHM[i,j] = ioniseR1[i,j] - ioniseR0[i,j]
        ioniseFWHMerr[i,j] = fn.findPositionErr(
            Rprofile, ioniseRate[i,:,j], ioniseErr[i,:,j],
            0.5, x0=ioniseR0[i,j], x1=ioniseR1[i,j], kind='linear'
        )

        ### find the neutral maximum and location
        neutralMax[i,j] = neutralDensity[i,:,j].max()
        neutralMaxR[i,j] = Rprofile[neutralDensity[i,:,j].argmax()]

        ### calculate the ionisation width, 1 e-folding length
        neutralR1[i,j], neutralR1A[i,j] = fn.findPosition(
            Rprofile, neutralDensity[i,:,j], -1., kind='exp'
        )
        neutralWidth1[i,j] = neutralMaxR[i,j] - neutralR1[i,j]
        neutralWidth1A[i,j] = neutralR1A[i,j] - neutralMaxR[i,j]
        neutralWidth1err[i,j], neutralWidth1Aerr[i,j] = fn.findPositionErr(
            Rprofile, neutralDensity[i,:,j], neutralErr[i,:,j], -1.,
            x0=neutralR1[i,j], x1=neutralR1A[i,j], kind='exp'
        )

        ### calculate the ionisation width, 2 e-folding lengths
        neutralR2[i,j], neutralR2A[i,j] = fn.findPosition(
            Rprofile, neutralDensity[i,:,j], -2., kind='exp'
        )
        neutralWidth2[i,j] = neutralMaxR[i,j] - neutralR2[i,j]
        neutralWidth2A[i,j] = neutralR2A[i,j] - neutralMaxR[i,j]
        neutralWidth2err[i,j], neutralWidth2Aerr[i,j] = fn.findPositionErr(
            Rprofile, neutralDensity[i,:,j], neutralErr[i,:,j], -2.,
            x0=neutralR2[i,j], x1=neutralR2A[i,j], kind='exp'
        )

        ### calculate the ionisation width, 3 e-folding lengths
        neutralR3[i,j], neutralR3A[i,j] = fn.findPosition(
            Rprofile, neutralDensity[i,:,j], -3., kind='exp'
        )
        neutralWidth3[i,j] = neutralMaxR[i,j] - neutralR3[i,j]
        neutralWidth3A[i,j] = neutralR3A[i,j] - neutralMaxR[i,j]
        neutralWidth3err[i,j], neutralWidth3Aerr[i,j] = fn.findPositionErr(
            Rprofile, neutralDensity[i,:,j], neutralErr[i,:,j], -3.,
            x0=neutralR3[i,j], x1=neutralR3A[i,j], kind='exp'
        )

        ### try loops to catch issues with loading HSV data
        try:
            ### get the fig density
            figPressure = HSV.calcFigPressure(shotn, time[i], fig='mid')
            figN0[i,j] = HSV.calcFigDensity(figPressure, T=300.)
        except:
            print('Unable to get fig pressure')
        try:
            ### get equilibrium data
            psiN[i,:,j] = HSV.getPsiN(shotn, time[i], Rprofile, 0.)[0][0,0,:]

            ### get the separatrix n0 / ne ratio
            sepInd = fn.findNearest(psiN[i,:,j], 1.)
            neutralRatio[i,j] = (neutralDensity[i,:,j] / profileDensity[i,:,j])[sepInd]
        except:
            print('Unable to get psiN and/or neutral Ratio')

TypeError: neutrals() missing 2 required positional arguments: 'fIonise' and 'scaleIonise'

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [ ]:
cmap = plt.get_cmap('RdBu_r')
norm = mcolors.Normalize(vmin=dR[0]*100., vmax=dR[-1]*100.)
sm = cm.ScalarMappable(cmap=cmap, norm=norm)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,3), dpi=150)

for i in range(0, NdR):
    if i != (NdR//2):
        ax.plot(Rprofile, profileDensity[0,:,i], color=cmap(norm(dR[i]*100.)))
ax.plot(Rprofile, profileDensity[0,:,NdR//2], color='k')
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('$\\mathrm{d}R$ (cm)', fontsize=9)
cbar.ax.tick_params(labelsize=9)

ax.set_xlabel('$R$ (m)', fontsize=9)
ax.set_ylabel('$n_e$ (m$^{-3}$)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.yaxis.get_offset_text().set_size(9)

ax.set_xlim([1.25,1.5])
ax.set_ylim([0., profileDensity[0].max()*1.1])
ax.set_title('electron density vs radius for different shifts', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,3), dpi=150)

for i in range(0, NdR):
    if i != (NdR//2):
        ax.plot(Rprofile, profileTemp[0,:,i], color=cmap(norm(dR[i]*100.)))
ax.plot(Rprofile, profileTemp[0,:,NdR//2], color='k')
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('$\\mathrm{d}R$ (cm)', fontsize=9)
cbar.ax.tick_params(labelsize=9)

ax.set_xlabel('$R$ (m)', fontsize=9)
ax.set_ylabel('$T_e$ (eV)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.yaxis.get_offset_text().set_size(9)

ax.set_xlim([1.25,1.5])
ax.set_ylim([0., profileTemp[0].max()*1.1])
ax.set_title('electron temperature vs radius for different shifts', fontsize=9)

plt.tight_layout()

In [ ]:
Rend = fn.findNearest(Rprofile, 1.5) + 1

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,3), dpi=150)

for i in range(0, NdR):
    if i != (NdR//2):
        ax.plot(Rprofile, ioniseRate[0,:,i], color=cmap(norm(dR[i]*100.)))
ax.plot(Rprofile, ioniseRate[0,:,NdR//2], color='k')
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('$\\mathrm{d}R$ (cm)', fontsize=9)
cbar.ax.tick_params(labelsize=9)

ax.set_xlabel('$R$ (m)', fontsize=9)
ax.set_ylabel('$S_\\mathrm{iz}$ (m$^{-3}$ s$^{-1}$)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.yaxis.get_offset_text().set_size(9)
ax.set_xlim([1.25,1.5])
ax.set_ylim([0., (ioniseRate[0])[:,:Rend].max()*1.1])
ax.set_title('ionisation rate vs radius for different shifts', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,3), dpi=150)

for i in range(0, NdR):
    if i != (NdR//2):
        ax.plot(Rprofile, neutralDensity[0,:,i], color=cmap(norm(dR[i]*100.)))
ax.plot(Rprofile, neutralDensity[0,:,NdR//2], '-k')
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('$\\mathrm{d}R$ (cm)', fontsize=9)
cbar.ax.tick_params(labelsize=9)

ax.set_xlabel('$R$ (m)', fontsize=9)
ax.set_ylabel('$n_0$ (m$^{-3}$)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.yaxis.get_offset_text().set_size(9)
ax.set_xlim([1.25,1.5])
ax.set_ylim([1e14, (neutralDensity[0])[:,:Rend].max()*2.])
ax.set_yscale('log')
ax.set_title('neutral density vs radius for different shifts', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,3), dpi=150)

ax.plot(Rprofile, emissivity[0,Rind:], '-', c='C0')
ax.fill_between(
    Rprofile, emissivity[0,Rind:]-emissivityErr[0,Rind:], 
    emissivity[0,Rind:]+emissivityErr[0,Rind:], color='C0', alpha=0.2
)

ax.set_xlabel('$R$ (m)', fontsize=9)
ax.set_ylabel('$\\epsilon$ (m$^{-3}$ s$^{-1}$)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.yaxis.get_offset_text().set_size(9)
ax.set_xlim([1.25,1.5])
ax.set_ylim([0., (emissivity+emissivityErr)[0,Rind:].max()*1.1])
ax.set_title('emissivity rate vs radius', fontsize=9)

plt.tight_layout()




fig, ax = plt.subplots(1, 1, figsize=(4,3), dpi=150)

ax.plot(Rprofile, ioniseRate[0,:,NdR//2], '-', c='C1')
ax.fill_between(
    Rprofile, ioniseRate[0,:,NdR//2]-ioniseErr[0,:,NdR//2], 
    ioniseRate[0,:,NdR//2]+ioniseErr[0,:,NdR//2], color='C1', alpha=0.2
)

ax.set_xlabel('$R$ (m)', fontsize=9)
ax.set_ylabel('$S_\\mathrm{iz}$ (m$^{-3}$ s$^{-1}$)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.yaxis.get_offset_text().set_size(9)
ax.set_xlim([1.25,1.5])
ax.set_ylim([0., (ioniseRate+ioniseErr)[0,:Rend,NdR//2].max()*1.1])
ax.set_title('ionisation rate vs radius for no shift', fontsize=9)

plt.tight_layout()




fig, ax = plt.subplots(1, 1, figsize=(4,3), dpi=150)

ax.plot(Rprofile, neutralDensity[0,:,NdR//2], '-', c='C2')
ax.fill_between(
    Rprofile, neutralDensity[0,:,NdR//2]-neutralErr[0,:,NdR//2], 
    neutralDensity[0,:,NdR//2]+neutralErr[0,:,NdR//2], color='C2', alpha=0.2)

ax.set_xlabel('$R$ (m)', fontsize=9)
ax.set_ylabel('$n_0$ (m$^{-3}$)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.yaxis.get_offset_text().set_size(9)
ax.set_xlim([1.25,1.5])
ax.set_ylim([1e14, (neutralDensity+neutralErr)[0,:Rend,NdR//2].max()*2.])
ax.set_yscale('log')
ax.set_title('neutral density vs radius for no shift', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5,3), dpi=150)

ax[0].plot(dR*100., ioniseMax[0], 'o', c='C1', mec='k')
ax[0].set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax[0].set_ylabel(
    '$S_\\mathrm{iz,max}$ (m$^{-3}$ s$^{-1}$)', fontsize=9
)

ax[0].tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax[0].minorticks_on()
ax[0].yaxis.get_offset_text().set_size(9)
ylim = ax[0].get_ylim()
ax[0].set_ylim(ylim)
ax[0].plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)

ax[1].plot(dR*100., ioniseMax[0], 'o', c='C1', mec='k')
ax[1].set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax[1].set_ylabel(
    '$S_\\mathrm{iz,max}$ (m$^{-3}$ s$^{-1}$)', fontsize=9
)

ax[1].set_yscale('log')
ax[1].tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax[1].minorticks_on()
ylim = ax[1].get_ylim()
ax[1].set_ylim(ylim)
ax[1].plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)
fig.suptitle('maximum ionisation rate vs radial shift', fontsize=10)

plt.tight_layout()

#############################################################

fig, ax = plt.subplots(1, 1, figsize=(3.5,3), dpi=150)

ax.plot(dR*100., ioniseMaxR[0], 'o', mec='C1', mfc='None')
ax.set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax.set_ylabel('$R(S_\\mathrm{iz,max})$ (m)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ylim = ax.get_ylim()
ax.set_ylim(ylim)
ax.plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)
ax.set_title('maximum ionisation location vs radial shift', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5,3), dpi=150)

ax[0].plot(dR*100., neutralMax[0], 'd', c='C2', mec='k')
ax[0].set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax[0].set_ylabel(
    '$n_{0,\\mathrm{max}}$ (m$^{-3}$)', fontsize=9
)

ax[0].tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax[0].minorticks_on()
ax[0].yaxis.get_offset_text().set_size(9)
ylim = ax[0].get_ylim()
ax[0].set_ylim(ylim)
ax[0].plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)

ax[1].plot(dR*100., neutralMax[0], 'd', c='C2', mec='k')
ax[1].set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax[1].set_ylabel(
    '$n_{0,\\mathrm{max}}$ (m$^{-3}$)', fontsize=9
)

ax[1].set_yscale('log')
ax[1].tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax[1].minorticks_on()
ylim = ax[1].get_ylim()
ax[1].set_ylim(ylim)
ax[1].plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)
fig.suptitle('maximum neutral denisty vs radial shift', fontsize=10)

plt.tight_layout()

#############################################################

fig, ax = plt.subplots(1, 1, figsize=(3.5,3), dpi=150)

ax.plot(dR*100., neutralMaxR[0], 'd', mec='C2', mfc='None')
ax.set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax.set_ylabel('$R(S_\\mathrm{iz,max})$ (m)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ylim = ax.get_ylim()
ax.set_ylim(ylim)
ax.plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)
ax.set_title('maximum neutral location vs radial shift', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5,3), dpi=150)

ax.errorbar(
    dR*100., ioniseFWHM[0]*100., yerr=ioniseFWHMerr[0]*100., 
    fmt='o', c='C1', mec='C1', mfc='None', capsize=2
)
ax.set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax.set_ylabel('FWHM $(S_\\mathrm{iz,max})$ (cm)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
# ylim = ax.get_ylim()
ylim = [0.,(ioniseFWHM[0]+ioniseFWHMerr[0]).max()*110.]
ax.set_ylim(ylim)
ax.plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)
ax.set_title('FWHM ionisation vs radial shift', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5,3), dpi=150)

ax.errorbar(
    dR*100., neutralWidth1[0]*100., yerr=neutralWidth1err[0]*100., 
    fmt='o', c='C0', mec='C0', mfc='None', capsize=2, label='1e'
)
ax.errorbar(
    dR*100., neutralWidth2[0]*100., yerr=neutralWidth2err[0]*100., 
    fmt='^', c='C1', mec='C1', mfc='None', capsize=2, label='2e'
)
ax.errorbar(
    dR*100., neutralWidth3[0]*100., yerr=neutralWidth3err[0]*100., 
    fmt='s', c='C2', mec='C2', mfc='None', capsize=2, label='3e'
)
ax.legend(fancybox=1, framealpha=1, fontsize=8)
ax.set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax.set_ylabel('FWHM $(n_{0,\\mathrm{max}})$ (cm)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
# ylim = ax.get_ylim()
ylim = [0.,(neutralWidth3[0]+neutralWidth3err[0]).max()*110.]
ax.set_ylim(ylim)
ax.plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)
ax.set_title('neutral density decay vs radial shift', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5,3), dpi=150)

ax[0].plot(dR*100., neutralRatio[0], 'X', c='C3', mec='C3', mfc='None')
ax[0].set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax[0].set_ylabel('$(n_0 / n_e)_\\mathrm{sep}$', fontsize=9)

ax[0].tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax[0].minorticks_on()
ylim = ax[0].get_ylim()
ylim = [0.,neutralRatio[0].max()*1.1]
ax[0].set_ylim(ylim)
ax[0].plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)

ax[1].plot(dR*100., neutralRatio[0], 'X', c='C3', mec='C3', mfc='None')
ax[1].set_xlabel('$\\mathrm{d}R$ (cm)', fontsize=9)
ax[1].set_ylabel('$(n_0 / n_e)_\\mathrm{sep}$', fontsize=9)

ax[1].tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax[1].minorticks_on()
ax[1].set_yscale('log')
ylim = ax[1].get_ylim()
ax[1].set_ylim(ylim)
ax[1].plot([0.,0.], ylim, '-k', lw=0.8, zorder=0)

fig.suptitle('neutral-to-electron density ratio vs radial shift', fontsize=9)

plt.tight_layout()

In [ ]:
loaded_dict = np.load('/home/pryan/Tesep/50499.npy', allow_pickle=True).item()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5,3), dpi=150)

ax.plot(loaded_dict['time'][3:-3], loaded_dict['Rsep_efit'][3:-3], '-', c='C0', lw=0.8, label='$R_\\mathrm{sep}$: EFIT')
ax.plot(loaded_dict['time'][3:-3], loaded_dict['Rsep'][3:-3], 'o', c='C0', mfc='None', label='$R_\\mathrm{sep}$: PRyan')

ax.legend(fancybox=1, framealpha=1, fontsize=8)
ax.set_xlabel('time $t$ (s)', fontsize=9)
ax.set_ylabel('Separatrix location - $R_\\mathrm{sep}$ (m)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.set_title('Separatrix vs time', fontsize=9)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5,3), dpi=150)

ax.plot(
    loaded_dict['time'][:-3], 
    100. * (loaded_dict['Rsep_efit'] - loaded_dict['Rsep'])[:-3], 
    'P', c='C3', mfc='None'
)

ax.set_xlabel('time $t$ (s)', fontsize=9)
ax.set_ylabel('$R_\\mathrm{sep}$: EFIT - $R_\\mathrm{sep}$: PRyan (cm)', fontsize=9)

ax.tick_params(
    axis="both", which='both', labelsize=9, direction='in', 
    left=True, bottom=True, right=True, top=True
)
ax.minorticks_on()
ax.set_title('Separatrix vs time', fontsize=9)

xlim = ax.get_xlim()
ax.set_xlim(xlim)
ax.plot(xlim, [0.,0.], '-k', lw=0.8, zorder=0)

plt.tight_layout()

In [ ]:
vignette = HSV.transform(HSV.vignette(), 50499)[239,32:216]
vignette2 = vignette / (152.067 + 121.7288)
vignette3 = (vignette - 121.7288) / (152.067 + 121.7288)
vignette3 = (vignette - 121.7288) / (152.067)
vignette4 = HSV.transform(HSV.vignette(amp=1., offset=0.), 50499)[239,32:216]
plt.figure()
# plt.plot(vignette)
plt.plot(vignette2)
plt.plot(vignette3)
plt.plot(vignette4, '--')

In [ ]:
np.isclose(vignette3, vignette4)